In [1]:
link_api_data_sus = "https://apidadosabertos.saude.gov.br/vigilancia-e-meio-ambiente/sistema-de-informacao-sobre-mortalidade"

In [2]:
import os, sys, requests, pprint
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql import functions as F

In [3]:
# Cria a conexão Spark

# 1. Remove qualquer barreira de proxy local que jogue o tráfego para a rede da empresa
os.environ.pop('HTTP_PROXY', None)
os.environ.pop('HTTPS_PROXY', None)
os.environ.pop('http_proxy', None)
os.environ.pop('https_proxy', None)

# 2. Garante que o Spark use o Python correto do venv
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# 3. Força o IP local estrito
os.environ['SPARK_LOCAL_IP'] = '127.0.0.1'

# 4. Inicializa configurando a autenticação local do Worker
spark = SparkSession.builder \
    .appName("TesteLocal") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.network.auth.enabled", "false") \
    .getOrCreate()

c:\Marco Conti\Projetos\MAIS-v2\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [14]:
url = "https://apidadosabertos.saude.gov.br/vigilancia-e-meio-ambiente/sistema-de-informacao-sobre-mortalidade?limit=100&offset=0"
response = requests.get(
    url,
    params={
        "limit": 1000,
        "offset": 0
    }
)

response.json()

{'sim': [{'racacor': '1',
   'dtconinv': None,
   'gravidez': None,
   'peso': None,
   'fonteinv': None,
   'dtrecebim': '29012026',
   'idademae': None,
   'dtrecoriga': '29122025',
   'atestante': '4',
   'comunsvoim': '355030',
   'horaobito': '1635',
   'contador': '2',
   'linhaii': '*E149',
   'altcausa': None,
   'causabas': 'N10',
   'versaosist': '3.2.30',
   'idade': '471',
   'natural': '850',
   'morteparto': None,
   'ocup': '223115',
   'acidtrab': None,
   'tp_altera': None,
   'atestado': 'N10X/ / / /E149',
   'difdata': '068',
   'necropsia': '1',
   'codmunres': '355030',
   'obitopuerp': None,
   'dtinvestig': None,
   'tppos': None,
   'qtdfilmort': None,
   'fonte': None,
   'seriescfal': None,
   'dtcadinf': None,
   'ocupmae': None,
   'escmaeagr1': None,
   'opor_do': '37',
   'fontes': None,
   'codmunnatu': '500270',
   'assistmed': None,
   'codestab': '2089599',
   'linhab': None,
   'obitoparto': None,
   'escmae2010': None,
   'dtcadinv': None,
   'causam

In [18]:

url = "https://apidadosabertos.saude.gov.br/vigilancia-e-meio-ambiente/sistema-de-informacao-sobre-mortalidade"

limit = 100
offset = 0

todos = []

while True:
    try:
        response = requests.get(
            url,
            params={
                "limit": limit,
                "offset": offset
            }
        )

        data = response.json()

        registros = data.get("sim", [])

        if not registros: # or offset >= 1000:
            break

        todos.extend(registros)

        print(f"Offset={offset} | Recebidos={len(registros)}", end="\r")

        offset += limit
    except Exception as e:
        print("offset:", offset, "Erro: ", str(e)[0:400])


KeyboardInterrupt: 

In [19]:
campos = registros[0].keys()

schema = StructType([
    StructField(campo, StringType(), True)
    for campo in campos
])

# Cria DataFrame Spark
df = spark.createDataFrame(todos, schema)

df.show(5, truncate=False)

+-------+--------+--------+----+--------+---------+--------+----------+---------+----------+---------+--------+---------------+--------+--------+----------+-----+-------+----------+------+--------+---------+----------------------------------+-------+---------+---------+----------+----------+-----+----------+-----+----------+--------+-------+----------+-------+------+----------+---------+--------+------+----------+----------+--------+--------+----------+------+----------+---------+----------+----------+----------+-----+---------+------+---------+------+------+--------+------+----------+----------+----------+--------+--------+----------+-----+-------+----+----------+----------+------+-------+---------+--------+----------+----------+----------+--------+----------+------+----------+----------+----+----------+--------+
|racacor|dtconinv|gravidez|peso|fonteinv|dtrecebim|idademae|dtrecoriga|atestante|comunsvoim|horaobito|contador|linhaii        |altcausa|causabas|versaosist|idade|natural|mort

In [20]:
df.count()
# df.printSchema()

206200

In [7]:
df.show(100,truncate=False)

+------+---------+----------+-------+------+---------+----------+----------+----------+----------+-------+----------+-------+----------+-------+----------+-----+----------+----------+-------+---------+----------+---------+---------+---------+----------+----------+----------+----------+---------+--------+----------+--------+--------+------+----------+----------+--------+---------+-------+---------+--------+----------+--------+--------+----------+----------+--------+-------+-----+---------+--------+-------+------+----------+---------+----------+-------+--------+----------+----------+---+---------+------------+--------+-----+-------+--------+--------+------+------+----------+----------+-------+---------+----------+----------+----+---------+--------+---------+---------+-----+----------+----------+----------+----------+----------+-------+----------+----------+----------+----------+---------+----------+----------+---------+--------+---------+----------+----------+----------+----------+------

In [9]:


# df.select("tp_not", "dt_notific", "nu_ano", "sg_uf").show(5, truncate=False)

df = (
    df.withColumn("idade_unidade", F.substring("nu_idade_n", 1, 1))
      .withColumn("idade_valor", F.substring("nu_idade_n", 2, 3).cast("int"))
)

(df.select("dt_notific"
          ,"id_municip"
          ,"ano_nasc" 
          ,"cs_sexo"
          ,"cs_raca"
          ,"idade_unidade"
          ,"idade_valor"
          )
   .show(5, truncate=False))



+----------+----------+--------+-------+-------+-------------+-----------+
|dt_notific|id_municip|ano_nasc|cs_sexo|cs_raca|idade_unidade|idade_valor|
+----------+----------+--------+-------+-------+-------------+-----------+
|2024-02-06|320530    |2005    |M      |4      |4            |19         |
|2024-02-06|320480    |1990    |F      |2      |4            |33         |
|2024-02-06|320020    |2009    |M      |1      |4            |14         |
|2024-02-06|320500    |1989    |M      |4      |4            |34         |
|2024-02-06|320510    |2001    |M      |4      |4            |22         |
+----------+----------+--------+-------+-------+-------------+-----------+
only showing top 5 rows
